# 📚 AY15: Repaso examen II
## Índice
- [10 -- Serialización y Networking I](#s10)
- [11 -- Serialización y Networking II](#s11)
- [13 -- Interfaces gráficas I](#s13)
- [14 -- Interfaces gráficas II](#s14)
- [15 -- Tópicos avanzados](#s15)


# Semana 10: Serialización y Networking I

---
# Tema 1: Serialización con JSON

### 1.1 Conceptos
- **Serializar**: transformar el *estado* de un objeto en una representación que se puede guardar o transmitir (texto como JSON, o bytes como `pickle`).
- **Deserializar**: el proceso inverso, se obtiene un objeto **nuevo** (igual `==`, pero no el mismo `is`).
- **Codificar** (`encode`): pasar texto a *bytes* (por ejemplo con UTF-8). Es necesario para mandar JSON por la red.

### 1.2 JSON: tipos y funciones
| Python | JSON |
|---|---|
| `dict` | objeto `{}` |
| `list`, `tuple` | arreglo `[]` |
| `str` | string |
| `int`, `float` | número |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

**ADVERTENCIA:** Una **tupla vuelve como lista**. No se pueden serializar por defecto funciones, `set` ni instancias de clases propias. Las **llaves** de un `dict` siempre quedan como `str`.

| Función | Qué hace |
|---|---|
| `json.dumps(obj)` | objeto $\rightarrow$ `str` |
| `json.loads(texto)` | `str` $\rightarrow$ objeto |
| `json.dump(obj, archivo)` | escribe la serialización en un archivo de texto abierto |
| `json.load(archivo)` | lee la serialización desde un archivo abierto |

In [1]:
import json

tupla = (None, 1, 22.33, {"a": True, "b": False})
texto = json.dumps(tupla)
print(texto, type(texto))

recuperado = json.loads(texto)
print(recuperado, type(recuperado)) # ahora es una lista! por que?
print("¿iguales?", list(tupla) == recuperado, "| ¿mismo objeto?", tupla is recuperado)

[null, 1, 22.33, {"a": true, "b": false}] <class 'str'>
[None, 1, 22.33, {'a': True, 'b': False}] <class 'list'>
¿iguales? True | ¿mismo objeto? False


In [2]:
# Las llaves int se transforman en str
print(json.loads(json.dumps({1: "uno"})))

{'1': 'uno'}


In [4]:
import os
# dump / load con archivos (abrir con encoding utf-8)
ruta = "ejemplo.json"
with open(ruta, "w", encoding="utf-8") as file:
    json.dump([1, 2, 3], file)
with open(ruta, "r", encoding="utf-8") as file:
    print("Desde archivo:", json.load(file))

Desde archivo: [1, 2, 3]


### 1.3 Serialización personalizada: `JSONEncoder`
Para serializar una instancia propia hay que convertirla en algo compatible (normalmente un `dict`).
- Opción rápida: `json.dumps(obj.__dict__)` $\rightarrow$ al deserializar obtienes un `dict`, **no** la instancia.
- Opción completa: heredar de `json.JSONEncoder` y sobrescribir `default(self, obj)`:
  - si reconoces el tipo $\rightarrow$ retornas un `dict` (conviene agregar un **discriminador** como `"tipo": "Mascota"`)
  - si no $\rightarrow$ `return super().default(obj)`, que levanta `TypeError`.
- Se usa con `json.dumps(obj, cls=MiEncoder)`. Un mismo encoder puede manejar varias clases (varios `isinstance`).

In [ ]:
class Mascota:
    def __init__(self, nombre: str, edad: int, especie: str, id_m: int) -> None:
        self.nombre = nombre
        self.edad = edad
        self.especie = especie
        self.id = id_m


class MascotaEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Mascota):
            return { # para reconstruir después
                "tipo": "Mascota",
                "Mascota_id": obj.id_,
                "nombre": obj.nombre,
                "edad": obj.edad,
                "especie": obj.especie,
            }
        return super().default(obj) # tipo desconocido -> TypeError


pepa = Mascota("Pepa", 70, "Tortuga", 1)
luna = Mascota("Luna", 8, "Gato", 0)
print(json.dumps(pepa.__dict__)) # solo atributos
print(json.dumps([pepa, luna], cls=MascotaEncoder)) # con encoder

try:
    json.dumps(object(), cls=MascotaEncoder)
except TypeError as error:
    print("TypeError:", error)

### 1.4 Deserialización personalizada: `object_hook`
- `json.loads(texto, object_hook=funcion)` llama a `funcion(diccionario)` por **cada objeto JSON** `{...}` (no para listas, strings, números ni `null`).
- Se procesa **de adentro hacia afuera**: el resultado del objeto interno pasa a formar parte del diccionario externo.
- La función debe reconocer *solo* los diccionarios que le corresponden (gracias al discriminador) y **retornar sin cambios** los demás.

In [ ]:
def mascota_hook(diccionario: dict):
    if diccionario.get("tipo") != "Mascota":
        return diccionario # no es mío lo devuelvo igual
    return Mascota(diccionario["nombre"], diccionario["edad"],
                   diccionario["especie"], diccionario["Mascota_id"])


texto = json.dumps({"dueña": "Ana", "mascotas": [pepa, luna]}, cls=MascotaEncoder)
datos = json.loads(texto, object_hook=mascota_hook)
print(datos["dueña"], [(m.nombre, type(m).__name__) for m in datos["mascotas"]])


# Orden de ejecución de adentro hacia afuera
def hook_que_imprime(diccionario):
    print("  hook recibió:", diccionario)
    return len(diccionario)

print(json.loads('{"a": {"b": 1, "c": {"d": 2}}}', object_hook=hook_que_imprime))

### 1.5 Errores al deserializar
Si el texto no es JSON válido, `loads`/`load` levantan `json.JSONDecodeError` (indica línea, columna y posición).

In [ ]:
for texto_malo in ['["sin cerrar"', '{"a": 1 "b": 2}', '["Luna", "Cachirulo" "Guillermo"]']:
    try:
        json.loads(texto_malo)
    except json.JSONDecodeError as error:
        print(f"JSON inválido -> {error}")

#### ✅ Ideas clave para recordar
- JSON es **texto**, independiente del lenguaje y legible, para mandarlo por red hay que hacer `.encode('utf-8')`.
- `dumps/loads` trabajan con `str`, `dump/load` con archivos abiertos en modo texto.
- Clase propia $\rightarrow$ `JSONEncoder.default` + `cls=`, de vuelta $\rightarrow$ `object_hook` + discriminador.
- `object_hook` se aplica a cada objeto `{}` empezando por los más internos.

---
# Tema 2: Conceptos básicos de redes (*networking*)

### 2.1 ¿Qué se necesita para que dos procesos se comuniquen?
1. Un **medio** físico (Ethernet, fibra, Wi-Fi, red móvil).
2. Una forma de identificar la **interfaz** de destino $\rightarrow$ **dirección IP**.
3. Una forma de identificar el **proceso** de destino $\rightarrow$ **puerto**.
4. Un **protocolo de transporte** $\rightarrow$ TCP o UDP.
5. Un **protocolo de aplicación** que da significado y formato a los mensajes $\rightarrow$ HTTP, DNS, SMTP… (¡o uno inventado por nosotros!).

### 2.2 IP, DNS y puertos
- **IP**: identifica una interfaz de red. IPv4 = 32 bits (`192.0.2.10`, $2^{32}$ combinaciones), IPv6 = 128 bits (`2001:db8::7334`). Privada $\rightarrow$ única en la red local, pública $\rightarrow$ única en Internet.
- **DNS** (*Domain Name System*): traduce nombres (`cursos.canvas.uc.cl`) a IPs, es distribuido y jerárquico.
- **Puerto**: número de 16 bits (0–65535) que identifica un punto de comunicación de la capa de transporte. TCP y UDP pueden usar el mismo número a la vez.

| Puerto | Servicio |
|---:|---|
| 21 | FTP (control) | 
| 22 | SSH | 
| 25 | SMTP |
| 53 | DNS | 
| 80 | HTTP |
| 110 | POP3 |
| 143 | IMAP |
| 443 | HTTPS |

Para contactar un servicio se necesita: **host + protocolo de transporte + puerto**.

### 2.3 TCP vs UDP
| | **TCP** (*Transmission Control Protocol*) | **UDP** (*User Datagram Protocol*) |
|---|---|---|
| Conexión | Sí (*handshake*) | No |
| Garantías | Flujo **ordenado y confiable** de bytes (secuencia, ACK, retransmisión, control de flujo y congestión) | Ni entrega, ni orden, ni retransmisión |
| Límites de mensaje | **No** los conserva (es un *stream*) | Sí los conserva |
| Usos | HTTP/1.1, HTTP/2, SMTP | DNS, voz/video en tiempo real, juegos, QUIC (HTTP/3) |

> 💡 Que TCP no conserve los límites de los mensajes es la razón por la que, más adelante, necesitamos **un protocolo propio** (enviar el largo antes del mensaje).

### 2.4 Encapsulamiento
Cada capa agrega su información de control al bajar por la pila y la quita al subir en el destino (TCP/UDP llevan los **puertos**, IP lleva las **direcciones**). Nuestros programas viven en la **capa de aplicación** y usan *sockets* que entrega el sistema operativo.

### 2.5 Arquitectura cliente-servidor
- **Servidor**: ofrece una funcionalidad y escucha en una dirección y puerto.
- **Cliente(s)**: consumen esa funcionalidad. En TCP el cliente *inicia la conexión*, en UDP solo envía datagramas.
- Ambos pueden estar en la misma máquina usando `localhost`.

#### ✅ Ideas clave para recordar
- IP identifica la máquina (interfaz), el puerto identifica el proceso.
- TCP = confiable, ordenado, con conexión, pero **sin límites de mensaje**. UDP = rápido, sin garantías, conserva límites.
- El protocolo de aplicación define *qué significan* los bytes.

---
# Tema 3: Servicios web: API, URL, HTTP

### 3.1 Servicio web y API
- **Servicio web**: programas que intercambian información usando tecnologías web (normalmente HTTP).
- **Sitio web**: entrega HTML/CSS/JS para una *persona* en un navegador. **API web**: expone datos y operaciones para *otros programas* (casi siempre en JSON).
- Una API documenta: URL base, **endpoints**, métodos HTTP permitidos, parámetros, cuerpo, autenticación, respuestas y códigos de estado.

### 3.2 URL
```text
https://api.ejemplo.com:8080/users/42/posts?page=2&limit=10#comentarios
└─┬─┘   └──────┬──────┘ └┬─┘└─────┬──────┘ └──────┬──────┘ └────┬────┘
esquema       host     puerto    ruta       query string     fragmento
```
- Puerto por defecto: 80 (HTTP), 443 (HTTPS). El **fragmento** no se envía al servidor.

### 3.3 HTTP
HTTP es un protocolo de **solicitud/respuesta** y es **stateless**: cada solicitud tiene sentido por sí sola (la *aplicación* puede guardar sesiones).

**Métodos**
| Método | Semántica |
|---|---|
| `GET` | Obtener una representación (debe ser *seguro*, no modifica) |
| `HEAD` | Como GET pero sin cuerpo |
| `POST` | Pedir que se procese el contenido (a menudo crea) |
| `PUT` | Crear o **reemplazar** el recurso |
| `PATCH` | Cambios **parciales** |
| `DELETE` | Eliminar |
| `OPTIONS` / `CONNECT` / `TRACE` | Opciones / túnel / diagnóstico |

**Códigos de estado**: 1xx informacional · **2xx éxito** · 3xx redirección · **4xx error del cliente** · **5xx error del servidor**.

Frecuentes: `200 OK`, `201 Created`, `202 Accepted`, `204 No Content`, `304 Not Modified`, `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `405 Method Not Allowed`, `409 Conflict`, `415 Unsupported Media Type`, `429 Too Many Requests`, `500 Internal Server Error`, `503 Service Unavailable`.

**Mensaje HTTP** = línea inicial + *headers* + cuerpo opcional. El header `Content-Type` indica el formato (`application/json`, `text/html`, `image/png`…).

#### ✅ Ideas clave para recordar
- API web = contrato entre programas (endpoints + métodos + formato).
- GET no modifica, POST procesa/crea, PUT reemplaza, PATCH modifica parcialmente.
- 4xx = la culpa es del cliente, 5xx = del servidor.

---
# Tema 4: Servicios web del lado del cliente (`requests`)

### 4.1 Esqueleto de una consulta

In [ ]:
import requests
url = "..."
respuesta = requests.get(url, params=..., headers=..., timeout=10)
respuesta.status_code
datos = respuesta.json()

- **Siempre** usar `timeout` (evita esperas infinitas) y comprobar el estado antes de interpretar.
- Atributos útiles: `status_code`, `reason`, `url` (URL final codificada), `headers`, `json()`, `text`, `content` (bytes).

### 4.2 Parámetros
- **De ruta**: forman parte del camino $\rightarrow$ `/recipes/1`, `/image/400x200/82944b`.
- **De consulta** (*query string*): van después de `?` $\rightarrow$ se pasan con `params={...}` y `requests` los codifica (espacios, tildes…).

### 4.3 Formato de la respuesta
El código de estado **no** dice el formato del cuerpo. Revisar `respuesta.headers["Content-Type"]`: puede ser JSON, texto, HTML, una imagen (bytes en `.content`) o nada.

### 4.4 POST con JSON
`requests.post(url, json=payload)` serializa el dict y agrega `Content-Type: application/json`. En cambio `data=dict` envía un **formulario**.

### 4.5 Autenticación
Las credenciales viajan en *headers*, p. ej. `{"Authorization": f"Bearer {TOKEN}"}`. Un token es como una contraseña: **no** se imprime, ni se guarda en el notebook, ni se sube a Git $\rightarrow$ usar variables de entorno (`os.environ.get("GITHUB_TOKEN")`). Sin token, el endpoint responde `401`.

> La siguiente celda usa la API educativa **DummyJSON** y necesita Internet, si no hay conexión, solo muestra un aviso.

In [ ]:
import requests

BASE = "https://dummyjson.com/"

try:
    # GET con parámetros de consulta
    respuesta = requests.get(BASE + "recipes/search",
                             params={"q": "chicken", "limit": 3, "select": "name"},
                             timeout=10)
    respuesta.raise_for_status()
    print("URL codificada:", respuesta.url)
    print("Content-Type:", respuesta.headers.get("Content-Type"))
    print([receta["name"] for receta in respuesta.json()["recipes"]])

    # POST con cuerpo JSON
    nuevo = {"title": "Comida de Pepa", "price": 2233}
    respuesta = requests.post(BASE + "products/add", json=nuevo, timeout=10)
    print(respuesta.status_code, respuesta.reason, respuesta.json())

    # Endpoint protegido sin token -> 401
    print("Sin token:", requests.get(BASE + "auth/me", timeout=10).status_code)
except requests.RequestException as error:
    print("No fue posible conectarse (¿sin Internet?):", type(error).__name__)

#### ✅ Ideas clave para recordar
- `params=` $\rightarrow$ query string · `json=` $\rightarrow$ cuerpo JSON · `headers=` $\rightarrow$ metadatos/credenciales · `timeout=` $\rightarrow$ límite de espera.
- Primero `raise_for_status()` / `status_code`, después `Content-Type`, recién ahí `json()`.
- Nunca escribir tokens en el código.

---
# Tema 5: Servicios web del lado del servidor (Flask)

### 5.1 Conceptos
- El servidor recibe solicitudes, ejecuta lógica y genera respuestas, puede estar escrito en cualquier lenguaje si respeta el protocolo.

### 5.2 Flask en una tabla
| Necesito… | En Flask |
|---|---|
| Asociar función $\leftrightarrow$ ruta | `@app.route("/ruta", methods=["GET", "POST"])` |
| Saber el método | `request.method` |
| Parte variable de la ruta | `"/hi/<string:username>"` $\rightarrow$ parámetro de la función |
| Query string | `request.args.get("edad", default=0, type=int)` |
| Cuerpo JSON | `request.get_json()` (solo si el cliente mandó `Content-Type: application/json`) |
| Responder JSON | `return {"clave": valor}` |
| Responder con código | `return {"error": "..."}, 400` |
| Cortar con error | `abort(404)` |
| Control total | `Response(status=..., response=..., content_type=...)` |

> Regla: si la entrada del cliente es inválida, responder **4xx** (400 / 422), nunca dejar que una excepción se convierta en `500`.

La siguiente celda levanta un servidor Flask en un *thread* (puerto elegido por el sistema), lo prueba y lo apaga.

In [6]:
import logging
import threading

import requests
from flask import Flask, Response, abort
from flask import request as flask_req
from werkzeug.serving import make_server

logging.getLogger("werkzeug").setLevel(logging.ERROR)
app = Flask("resumen")


@app.route("/hi/<string:username>", methods=["GET"])          # caso 1: ruta dinámica
def saludar(username):
    return {"texto": f"Hola {username}!"}


@app.route("/argumentos", methods=["GET"])                      # caso 2: query string
def argumentos():
    nombre = flask_req.args.get("nombre", default="Guillermo", type=str)
    edad = flask_req.args.get("edad", default=0, type=int)
    return {"texto": f"{nombre} tiene {edad} años"}


@app.route("/body", methods=["POST"])                           # caso 3: JSON + validación
def sumar():
    datos = flask_req.get_json(silent=True)
    if not isinstance(datos, dict) or not {"var_1", "var_2"} <= datos.keys():
        return {"error": "Faltan var_1 o var_2."}, 400
    if not all(isinstance(datos[k], (int, float)) for k in ("var_1", "var_2")):
        return {"error": "Deben ser números."}, 422
    return {"result": datos["var_1"] + datos["var_2"]}


@app.route("/error/<int:instruccion>", methods=["DELETE"])     # caso 4: respuestas
def error(instruccion):
    if instruccion == 0:
        abort(400)
    if instruccion == 1:
        return Response(status=418)
    if instruccion == 2:
        return Response(status=200, response="Texto plano",
                        content_type="text/plain; charset=utf-8")
    abort(404)

In [7]:
servidor = make_server("localhost", 0, app)          # puerto 0 = el SO elige uno libre
hilo = threading.Thread(target=servidor.serve_forever, daemon=True)
hilo.start()
URL = f"http://localhost:{servidor.server_port}"

print(requests.get(f"{URL}/hi/Luna", timeout=5).json())
print(requests.get(f"{URL}/argumentos", params={"nombre": "Pepa", "edad": 70}, timeout=5).json())
print(requests.post(f"{URL}/body", json={"var_1": 11, "var_2": 13}, timeout=5).json())
r = requests.post(f"{URL}/body", json={"var_1": "a", "var_2": 1}, timeout=5)
print(r.status_code, r.json())
for i in range(4):
    r = requests.delete(f"{URL}/error/{i}", timeout=5)
    print(f"/error/{i} ->", r.status_code, r.text[:30].strip())

{'texto': 'Hola Luna!'}
{'texto': 'Pepa tiene 70 años'}
{'result': 24}
422 {'error': 'Deben ser números.'}
/error/0 -> 400 <!doctype html>
<html lang=en>
/error/1 -> 418 
/error/2 -> 200 Texto plano
/error/3 -> 404 <!doctype html>
<html lang=en>


In [8]:
servidor.shutdown()
servidor.server_close()
hilo.join(timeout=2)
print("Servidor detenido")

Servidor detenido


#### ✅ Ideas clave para recordar
- Flask = aplicación WSGI, Werkzeug (`app.run`/`make_server`) = servidor de desarrollo.
- Ruta dinámica $\rightarrow$ argumento de la función, query $\rightarrow$ `request.args`, JSON $\rightarrow$ `request.get_json()`.
- Validar la entrada y responder 400/422 en lugar de fallar con 500.

<a id="s11"></a>
# 🔌 Semana 11 — Serialización y *Networking* II

---
# Tema 7: Serialización con `pickle`

### 7.1 Funciones
`pickle` serializa **casi cualquier objeto de Python** a **bytes** (incluidas instancias de clases propias, sin encoder).
- `pickle.dumps(obj)` $\rightarrow$ `bytes` · `pickle.loads(b)` $\rightarrow$ objeto.
- `pickle.dump(obj, f)` / `pickle.load(f)` $\rightarrow$ archivos abiertos en **modo binario** (`"wb"`, `"rb"`).

⚠️ **`pickle` no es seguro**: cargar un pickle de procedencia desconocida puede ejecutar código malicioso.

### 7.2 Personalización
- Al serializar, si la clase define `__getstate__`, pickle guarda **lo que retorne** (un `dict`), si no, guarda `obj.__dict__`. Trabajar sobre una **copia** (`self.__dict__.copy()`) para no alterar el original.
- Al deserializar, si existe `__setstate__(self, state)`, se ejecuta y debe terminar con `self.__dict__ = state`, si no, se asigna directamente.
- Uso típico: **eliminar** al guardar atributos que no tienen sentido en otro programa (sockets, conexiones, locks) y **recrearlos** al cargar.

In [ ]:
import pickle
import threading


class Sala:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.mensajes = ["hola"]
        self.lock = threading.Lock() # no se puede serializar

    def __getstate__(self) -> dict:
        estado = self.__dict__.copy() # es una copia, no modificamos el original
        del estado["lock"]
        print("[__getstate__] guardando", list(estado))
        return estado

    def __setstate__(self, estado: dict) -> None:
        print("[__setstate__] restaurando y recreando el lock")
        estado["lock"] = threading.Lock()
        self.__dict__ = estado


original = Sala("general")
datos = pickle.dumps(original)
print(type(datos), datos[:20], "...")
copia = pickle.loads(datos)
print(copia.nombre, copia.mensajes, type(copia.lock).__name__)
print("¿mismo objeto?", copia is original, "| ¿el original conserva su lock?", hasattr(original, "lock"))

### 7.3 JSON vs pickle
| | JSON | pickle |
|---|---|---|
| Resultado | Texto (`str`) | `bytes` |
| Legible por humanos | Sí | No |
| Otros lenguajes | Sí | Solo Python |
| Clases propias | Requiere `JSONEncoder` / `object_hook` | Directo |
| Seguridad | Segura de cargar | **Insegura** con datos no confiables |
| Personalización | `default` / `object_hook` | `__getstate__` / `__setstate__` |

#### ✅ Ideas clave para recordar
- pickle $\rightarrow$ bytes, archivos en modo `b`.
- `__getstate__` decide qué se guarda, `__setstate__` cómo se reconstruye.
- No cargar pickles de terceros.

---
# Tema 8: Manejo de *bytes*

### 8.1 Encoding
- 1 *byte* = 8 *bits* $\rightarrow$ $2^8 = 256$ valores (0–255). Un **encoding** asocia números con caracteres (ASCII: 128 caracteres, **UTF-8** es compatible con ASCII y es el que usaremos).
- `ord("B") $\rightarrow$ 66`, `chr(66) $\rightarrow$ "B"`, `hex(255) $\rightarrow$ "0xff"`, `bin(104) $\rightarrow$ "0b1101000"`.

### 8.2 `bytes` (inmutable) y `bytearray` (mutable)
| | `bytes` | `bytearray` |
|---|---|---|
| Mutable | No | Sí (como una lista) |
| Crear | `b"abc"`, `bytes((97, 98))`, `"á".encode("utf-8")` | `bytearray(b"abc")`, `bytearray()` |
| Indexar `x[0]` | Retorna un **int** | Retorna un **int** |
| Agregar | — | `append(int)`, `extend(bytes)` |

- `texto.encode("utf-8")` $\rightarrow$ bytes, `b.decode("utf-8")` $\rightarrow$ str. Los mismos bytes decodificados con otro encoding dan **otro** texto.
- `encode(..., errors=)`: `"strict"` (por defecto, error), `"replace"` (`?`), `"ignore"`, `"xmlcharrefreplace"`.
- Un literal `b"..."` solo admite caracteres ASCII (`b"áb"` es error de sintaxis).

In [ ]:
texto = "estación"
codificado = texto.encode("utf-8")
print(codificado, len(texto), "caracteres ->", len(codificado), "bytes")
print(codificado.decode("utf-8"), "|", b"\x63\x6c\x69\x63\x68\xe9".decode("latin-1"))
print(texto.encode("ascii", errors="replace"), texto.encode("ascii", errors="ignore"))

ba = bytearray(b"holamundo")
ba[4:6] = b"MU"
ba.append(33)            # append recibe un int
ba.extend(b"??")         # extend recibe bytes
print(ba, ba[0], bin(ba[0])[2:].zfill(8))

### 8.3 *Chunks*
Un *chunk* es un grupo de bytes. Leer todo de una vez puede no caber en memoria, de a uno es lento $\rightarrow$ se lee **por bloques**.
```python
with open("archivo.bin", "rb") as archivo:
    while chunk := archivo.read(4096):   # walrus: asigna y evalúa, b"" termina el ciclo
        procesar(chunk)
```

### 8.4 Enteros $\leftrightarrow$ bytes y *endianness*
- `n.to_bytes(largo, byteorder)` y `int.from_bytes(b, byteorder)`.
- **Big endian**: byte más significativo **al inicio** (256 $\rightarrow$ `\x01\x00`). **Little endian**: al final (256 $\rightarrow$ `\x00\x01`).
- Si `largo` es mayor que lo necesario se rellena con ceros. Con 1 solo byte el orden no importa.

In [ ]:
datos = bytearray(b"Una gran cantidad de bytes")
print([bytes(datos[i:i + 8]) for i in range(0, len(datos), 8)])   # chunks de 8

print((256).to_bytes(2, "big"), (256).to_bytes(2, "little"), (256).to_bytes(4, "big"))
print(int.from_bytes(b"\x00\x00\x04\x00", "big"), int.from_bytes(b"\x00\x00\x04\x00", "little"))

# "No confíes en los print de bytes": 9 se imprime como \t y 13 como \r
print((9).to_bytes(1, "big"), (13).to_bytes(1, "big"), int.from_bytes(b"\t", "big"))

#### ✅ Ideas clave para recordar
- `bytes` inmutable, `bytearray` mutable, indexar ambos da `int`.
- Big endian = más significativo primero (es lo que usaremos en los protocolos).
- Para verificar bytes, convertirlos a `int` en vez de fiarse del `print`.

---
# Tema 9: *Networking* con Python: *sockets* TCP

### 9.1 *Socket*
Objeto del sistema operativo que permite a un programa enviar y recibir datos por la red: nuestro punto de entrada/salida.
```python
socket.socket(family, type)
#   family: AF_INET (IPv4) | AF_INET6 (IPv6)
#   type:   SOCK_STREAM (TCP) | SOCK_DGRAM (UDP)
```

### 9.2 Pasos TCP
| Servidor | Cliente |
|---|---|
| `sock = socket(AF_INET, SOCK_STREAM)` | `sock = socket(AF_INET, SOCK_STREAM)` |
| `sock.bind((host, puerto))` | (el SO asigna un puerto al cliente) |
| `sock.listen(n)` — *n* conexiones pendientes máx. | |
| `cliente, direccion = sock.accept()` ⏸️ bloquea | `sock.connect((host, puerto))` (acepta IP o nombre $\rightarrow$ DNS) |
| `cliente.recv(4096)` / `cliente.sendall(b)` | `sock.sendall(b)` / `sock.recv(4096)` |
| `cliente.close()` y `sock.close()` | `sock.close()` |

- `send(b)` puede enviar **solo una parte** y retorna cuántos bytes envió, `sendall(b)` reintenta hasta enviar todo (si falla, levanta excepción sin decir cuánto alcanzó a enviar).
- `recv(n)` retorna **a lo más** `n` bytes (los que hayan llegado), retorna `b""` si la otra parte cerró la conexión.
- Los sockets solo transmiten **bytes** $\rightarrow$ `encode`/`decode`, o serializar con JSON/pickle.
- `ConnectionError` es la base de `BrokenPipeError`, `ConnectionRefusedError`, `ConnectionResetError`, `ConnectionAbortedError`. Cerrar siempre en un `finally`.

In [ ]:
import socket
import threading

# --- Servidor "echo" en un thread: responde en mayúsculas lo que recibe ---
servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
servidor.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
servidor.bind(("localhost", 0)) # 0: el SO elige un puerto libre
servidor.listen()
PUERTO = servidor.getsockname()[1]

def atender():
    sock_cliente, direccion = servidor.accept()      # bloquea hasta que llegue alguien
    print("[Servidor] conexión desde", direccion)
    while data := sock_cliente.recv(4096):           # b"" = el cliente cerró
        sock_cliente.sendall(data.upper())
    sock_cliente.close()
    servidor.close()
    print("[Servidor] cerrado")

hilo_servidor = threading.Thread(target=atender, daemon=True)
hilo_servidor.start()

# --- Cliente ---
cliente = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
try:
    cliente.connect(("localhost", PUERTO))
    cliente.sendall("hola servidor".encode("utf-8"))
    print("[Cliente] recibí:", cliente.recv(4096).decode("utf-8"))
except ConnectionError:
    print("[Cliente] ocurrió un error de conexión")
finally:
    cliente.close()
hilo_servidor.join(timeout=2)

#### ✅ Ideas clave para recordar
- Servidor: `bind` $\rightarrow$ `listen` $\rightarrow$ `accept` (un socket nuevo por cliente). Cliente: `connect`.
- `recv(n)` entrega *hasta* n bytes, `b""` significa conexión cerrada.
- Preferir `sendall`, o `send` en un ciclo que sume lo enviado.

---
# Tema 10: Ejemplos de *networking*: protocolo, múltiples clientes y arquitecturas

### 10.1 Enviar datos serializados
- **JSON**: `sock.sendall(json.dumps(datos).encode("utf-8"))` $\rightarrow$ `json.loads(sock.recv(4096).decode("utf-8"))`.
- **pickle**: `sock.sendall(pickle.dumps(obj))` $\rightarrow$ `pickle.loads(sock.recv(4096))`.
- Al *echo server* no le importa el formato: solo recibe bytes y los devuelve.

### 10.2 El problema de los mensajes grandes
Con un solo `recv(4096)` un archivo de 100 KB llega **incompleto**: TCP es un flujo y no sabe dónde termina "un mensaje". Preguntas: ¿cuánto voy a recibir? ¿qué pasa con MB o GB?

**Solución = protocolo + chunks**
1. El emisor envía primero el **largo** en 4 bytes (`len(datos).to_bytes(4, "big")`) $\rightarrow$ mensajes de hasta $2^{32}$ bytes (4 GiB).
2. Luego envía los datos (con `sendall`, o con `send` en un ciclo `enviados += sock.send(datos[enviados:])`).
3. El receptor lee los 4 bytes, los convierte con `int.from_bytes` y sigue haciendo `recv(min(4096, faltante))` **hasta completar el largo**, contando lo que *realmente* retornó cada `recv`.

In [ ]:
import socket
import threading

TAMANO_CHUNK = 4096

def enviar(sock: socket.socket, mensaje: bytes) -> None:
    sock.sendall(len(mensaje).to_bytes(4, byteorder="big") + mensaje)

def recibir_bytes(sock: socket.socket, cantidad: int) -> bytes:
    datos = bytearray()
    while len(datos) < cantidad:
        chunk = sock.recv(min(TAMANO_CHUNK, cantidad - len(datos)))
        if not chunk:                                   # la otra parte cerró
            raise ConnectionError("La conexión se cerró")
        datos.extend(chunk)
    return bytes(datos)

def recibir(sock: socket.socket) -> bytes:
    largo = int.from_bytes(recibir_bytes(sock, 4), byteorder="big")
    return recibir_bytes(sock, largo)


# enviemos 1 MB y comprobemos que llega completo
receptor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
receptor.bind(("localhost", 0)), receptor.listen()
puerto = receptor.getsockname()[1]
resultado = {}

def recibir_archivo():
    sock_cliente, _ = receptor.accept()
    resultado["datos"] = recibir(sock_cliente)
    enviar(sock_cliente, "Gracias.".encode("utf-8"))
    sock_cliente.close(), receptor.close()

hilo = threading.Thread(target=recibir_archivo), hilo.start()

archivo = bytes(range(256)) * 4096                      # 1 MiB de datos
emisor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
emisor.connect(("localhost", puerto))
enviar(emisor, archivo)
print("Respuesta:", recibir(emisor).decode("utf-8"))
emisor.close(), hilo.join()
print(f"Enviados {len(archivo)} bytes, recibidos {len(resultado['datos'])}, ¿iguales? {archivo == resultado['datos']}")

### 10.3 Servidor con múltiples clientes concurrentes
`accept()` y `recv()` **bloquean** $\rightarrow$ se necesitan *threads*:
- **1 thread** que acepta clientes en un ciclo infinito.
- **1 thread por cliente** que escucha sus mensajes, llama a `handle_command` y responde.

```python
class Server:
    def __init__(self, port, host):
        self.socket_server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.socket_server.bind((host, port))
        self.socket_server.listen()
        threading.Thread(target=self.accept_connections_thread).start()

    def accept_connections_thread(self):
        while True:
            client_socket, _ = self.socket_server.accept()
            threading.Thread(target=self.listen_client_thread,
                             args=(client_socket,), daemon=True).start()

    def listen_client_thread(self, client_socket):
        while True:
            recibido = recibir(client_socket).decode()        # protocolo de 4 bytes
            respuesta = self.handle_command(recibido, client_socket)
            enviar(client_socket, str(respuesta).encode())
```
El **cliente** también usa un thread (`daemon=True`) para escuchar al servidor (así el servidor puede enviar mensajes sin que el cliente los pida) y el thread principal lee `input()` y envía (REPL).

> 💡 Mantener las clases `Client`/`Server` **solo para comunicación** y poner la lógica en otras clases (*backend*): si no, el código se vuelve difícil de leer y mantener.

### 10.4 Otras arquitecturas
- **Cliente conectado a varios servidores**: un socket **por servidor** (p. ej. `self.sockets = {1: socket(...), 2: socket(...)}`), cada servidor en su propio puerto y **un thread que escucha por cada conexión**.
- **P2P (*Peer to Peer*)**: sin servidor central, cada *peer* tiene un socket "servidor" (`bind`/`listen`/`accept`) y un socket "cliente" (`connect` al otro peer). Ojo con el orden de arranque (el primero debe estar escuchando).

**Receta general**
- Decidir quién es servidor y quién cliente (puede ser ambos).
- Un thread por cada socket que se escucha + un thread por cada cliente conectado.
- Threads extra según se necesite (p. ej. leer la terminal).

#### ✅ Ideas clave para recordar
- TCP no separa mensajes $\rightarrow$ **enviar el largo primero** y recibir por chunks hasta completarlo.
- Servidor multicliente = 1 thread que acepta + 1 thread por cliente.
- Separar red (comunicación) de lógica (*backend*).

# 🖼️ Semana 13 — Interfaces gráficas I (PyQt6)

⚠️ Jupyter no está hecho para ejecutar interfaces de escritorio: los ejemplos de PyQt se muestran como **texto**. Para probarlos, cópialos en un `.py` (o usa la carpeta `scripts/` de cada semana) y ejecútalos desde una terminal ubicada en esa carpeta.

---
# Tema 11: Interfaces gráficas y primera ventana en PyQt

### 11.1 Eventos vs *polling*
- *Polling*: un `while True` que pregunta constantemente a cada elemento si pasó algo $\rightarrow$ muy costoso.
- **Arquitectura basada en eventos**: un **evento** es una acción (clic, tecla, ventana cerrada…). Para cada evento `e` se define un **handler** (manejador) que se ejecuta cuando ocurre, de forma asíncrona.

### 11.2 PyQt6
*Framework* multiplataforma basado en la biblioteca C++ **Qt**. Módulos más usados:
| Módulo | Contiene |
|---|---|
| `QtWidgets` | Elementos clásicos de GUI (ventanas, botones, labels, layouts) |
| `QtCore` | Funcionalidades no gráficas: *event loop*, señales, `QTimer`, `QThread` |
| `QtGui` | Integración de ventanas, eventos, imágenes (`QPixmap`), fuentes, `QAction` |
| Otros | `QtNetwork`, `QtMultimedia`, `QtSql`, `QtSvg`, `QtOpenGL`, `QtBluetooth` |

### 11.3 Plantilla mínima
- Los elementos gráficos se llaman ***widgets***, `QWidget` es la clase base de todos (todos son rectangulares).
- `QApplication` contiene el *main loop*: **debe crearse antes que cualquier widget** y existe **una sola** por programa (`QApplication([])`).
- Un widget creado solo existe en memoria hasta llamar a `show()`. `app.exec()` inicia el *event loop* y retorna el código de salida.

```python
import sys
from PyQt6.QtWidgets import QApplication, QWidget


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.setGeometry(200, 100, 300, 300)   # (x, y, ancho, alto)
        self.setWindowTitle("Mi Primera Ventana")


if __name__ == "__main__":
    def hook(type_, value, traceback) -> None:
        # Imprime el error con el hook original, al instalar uno propio PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook

    app = QApplication([])
    ventana = MiVentana()
    ventana.show()
    sys.exit(app.exec())        # 0 = terminó sin errores
```

### 11.4 Depuración
En PyQt el programa puede caerse **sin mostrar el error**. El bloque `hook` de arriba hace que el *traceback* se imprima (se explica en detalle en el Tema 15).

#### ✅ Ideas clave para recordar
- Una sola `QApplication`, creada antes que los widgets.
- `show()` muestra, `app.exec()` inicia el event loop.
- Siempre instalar el hook de excepciones.

---
# Tema 12: Elementos gráficos básicos y *layouts*


### 12.1 Coordenadas
`setGeometry(x, y, ancho, alto)`: el origen `(0, 0)` es la **esquina superior izquierda** de la pantalla (para ventanas) o de la **ventana padre** (para widgets internos). **X crece a la derecha, Y hacia abajo.**

### 12.2 Widgets básicos
| Widget | Para qué | Métodos útiles |
|---|---|---|
| `QLabel(texto, padre)` | Mostrar texto (o una imagen) | `setText`, `text`, `setPixmap`, `setScaledContents`, `setStyleSheet` |
| `QLineEdit(texto, padre)` | Ingresar una línea de texto | `text()`, `setText` |
| `QPushButton(texto, padre)` | Botón (`"&Texto"` destaca la primera letra como atajo) | `clicked`, `setText` |
| `QPixmap(ruta)` (de `QtGui`) | Cargar los píxeles de una imagen | se carga en un `QLabel` |

- El segundo argumento (`self`) es el **widget padre**: sin padre, el widget **no aparece** dentro de la ventana.
- Posicionamiento absoluto: `move(x, y)`, `setGeometry(...)`, `resize(w, h)`, `resize(widget.sizeHint())` (tamaño sugerido).

```python
import os
from PyQt6.QtGui import QPixmap
from PyQt6.QtWidgets import QWidget, QLabel, QLineEdit, QPushButton


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.setGeometry(200, 100, 250, 250)
        self.label = QLabel("Texto:", self)
        self.label.move(10, 15)
        self.edit = QLineEdit("", self)
        self.edit.setGeometry(55, 15, 120, 20)
        self.boton = QPushButton("&Procesar", self)
        self.boton.resize(self.boton.sizeHint())
        self.boton.move(10, 45)

        self.imagen = QLabel(self)
        self.imagen.setGeometry(10, 80, 100, 100)
        self.imagen.setPixmap(QPixmap(os.path.join("img", "python.jpg")))
        self.imagen.setScaledContents(True)          # ajusta la imagen al tamaño del label
```

### 12.3 *Layouts*
El posicionamiento absoluto no se adapta al redimensionar la ventana ni a otras pantallas $\rightarrow$ usar *layouts*:
- `QHBoxLayout` (horizontal) y `QVBoxLayout` (vertical): `addWidget(w)`, `addLayout(l)` (se pueden **anidar**), `addStretch(k)` (espaciador con peso *k*).
- `QGridLayout`: grilla de filas y columnas, `addWidget(w, fila, columna)`.
- Se asignan a la ventana con `self.setLayout(layout)`.

```python
from PyQt6.QtWidgets import QHBoxLayout, QVBoxLayout, QGridLayout, QPushButton

# Dentro de init_gui (los widgets ya creados):
hbox = QHBoxLayout()
hbox.addStretch(1)
hbox.addWidget(self.label)
hbox.addWidget(self.edit)
hbox.addWidget(self.boton)
hbox.addStretch(1)

vbox = QVBoxLayout()
vbox.addStretch(5)            # empuja el hbox hacia abajo
vbox.addLayout(hbox)
vbox.addStretch(1)

grilla = QGridLayout()        # teclado de teléfono
valores = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "*", "0", "#"]
posiciones = [(i, j) for i in range(4) for j in range(3)]
for posicion, valor in zip(posiciones, valores):
    grilla.addWidget(QPushButton(valor, self), *posicion)
vbox.addLayout(grilla)

self.setLayout(vbox)
```

### 12.4 Widgets propios con POO
Heredar permite **reutilizar** código y agregar atributos/comportamiento:
```python
class MiBoton(QPushButton):
    def __init__(self, nombre: str, pos: tuple, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)        # QPushButton("Apriétame", padre)
        self.nombre = nombre
        self.contador = 0
        self.resize(self.sizeHint())
        self.move(*pos)
        self.clicked.connect(self.contar)        # cada instancia lleva su propio conteo

    def contar(self) -> None:
        self.contador += 1
        print(f"{self.nombre} apretado {self.contador} veces")


class CampoFormulario(QHBoxLayout):               # un campo = label + line edit
    def __init__(self, texto: str) -> None:
        super().__init__()
        self.addWidget(QLabel(f"{texto}: "))
        self.addWidget(QLineEdit(""))
# En el formulario: contenedor.addLayout(CampoFormulario("Nombre")), etc.
```

#### ✅ Ideas clave para recordar
- Origen arriba-izquierda, Y crece hacia abajo.
- Sin widget padre no se ve.
- Layouts > posiciones absolutas, se anidan y se asignan con `setLayout`.

---
# Tema 13: Eventos y señales

### 13.1 Modelo de eventos
- **Fuente**: objeto que genera el evento · **Objeto evento**: encapsula el cambio · **Destino**: objeto que se notifica.
- PyQt implementa esto con **signals** y **slots**: cuando ocurre algo, el objeto emite una *señal* que llama al *slot* conectado (cualquier *callable*).
- Los eventos se detectan una vez iniciado el *event loop* (`app.exec()`).

### 13.2 Señales predefinidas y `sender`
```python
boton.clicked.connect(self.boton_clickeado)          # sin paréntesis, se pasa la función
boton_salir.clicked.connect(QCoreApplication.instance().quit)

def boton_clickeado(self) -> None:
    boton = self.sender()                            # el objeto que emitió la señal
    idx = self.grilla.indexOf(boton)
    fila, columna, *_ = self.grilla.getItemPosition(idx)
    self.label.setText(f"Botón {boton.text()} en ({fila}, {columna})")
```
> Si un `setText` no se refleja de inmediato en algún sistema, se puede forzar con `label.repaint()`.

### 13.3 Eventos de mouse y teclado (se **sobrescriben**, no se conectan)
| Método de `QWidget` | Cuándo | Dato útil |
|---|---|---|
| `mousePressEvent(event)` | Se presiona el mouse | `event.position().x()`, `.y()`, `self.label.underMouse()` |
| `mouseReleaseEvent(event)` | Se suelta | ídem |
| `mouseMoveEvent(event)` | Se mueve **con botón presionado**, o siempre si `setMouseTracking(True)` | posición |
| `keyPressEvent(event)` | Se presiona una tecla | `event.text()`, `event.key()` (p. ej. `Qt.Key.Key_W`) |
| `keyReleaseEvent(event)` | Se suelta | ídem |
| Otros | `enterEvent`, `leaveEvent`, `dragEnterEvent`, `closeEvent` | |

### 13.4 Señales personalizadas
```python
from PyQt6.QtCore import pyqtSignal
from PyQt6.QtWidgets import QWidget, QLabel


class VentanaPresionable(QWidget):
    senal_simple = pyqtSignal()                 # sin datos
    senal_texto = pyqtSignal(str)               # envía un str
    senal_coordenadas = pyqtSignal(int, int)    # envía dos int (también dict, list, ...)

    def mousePressEvent(self, event) -> None:
        self.senal_simple.emit()
        self.senal_texto.emit("¡me presionaron!")

    def mouseMoveEvent(self, event) -> None:
        self.senal_coordenadas.emit(event.pos().x(), event.pos().y())


class VentanaQueSeEdita(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.etiqueta = QLabel("", self)

    def recibir_coordenadas(self, x: int, y: int) -> None:   # misma firma que la señal
        self.etiqueta.setText(f"Recibí {x}, {y}")


# En el main:
# emisora = VentanaPresionable(), receptora = VentanaQueSeEdita()
# emisora.senal_coordenadas.connect(receptora.recibir_coordenadas)
```
**Reglas de `pyqtSignal`** (por implementación interna de PyQt):
1. Debe vivir en una clase que herede de `QObject` (todo `QWidget` ya lo es, si no, heredar de `QObject` y llamar a `super().__init__()`).
2. Se define como **atributo de clase**, no de instancia… y aun así **cada instancia tiene su propia señal**.
3. El nombre del atributo solo puede usar caracteres ASCII (sin `ñ` ni tildes).
4. El *slot* conectado recibe como argumentos los datos de `emit`, en el mismo orden y tipos declarados.

#### ✅ Ideas clave para recordar
- Señales predefinidas (`clicked`) se **conectan**, eventos (`mousePressEvent`) se **sobrescriben**.
- `connect(funcion)` sin paréntesis.
- `pyqtSignal(tipos)` como atributo de clase de un `QObject`, `emit(datos)` $\rightarrow$ slot(datos).

---
# Tema 14: Diseño *front-end* / *back-end* y conexión entre ventanas


### 14.1 Cohesión y acoplamiento
- **Front-end**: la interfaz gráfica con que interactúa el usuario. **Back-end**: la lógica y los datos.
- **Cohesión**: cada componente hace **solo** las tareas para las que fue creado (delegando el resto). Queremos **alta**.
- **Acoplamiento**: cuánto obliga un cambio en un componente a cambiar otro. Queremos **bajo** (nunca es cero, pero se minimiza).
- Ventajas de separar: modularidad, no cargar a la interfaz con cálculos pesados, escalamiento, equipos con distinta experticia, *testing* por partes y poder cambiar una parte completa (p. ej. PyQt5 $\rightarrow$ PyQt6, o a web) sin tocar la otra.

### 14.2 El ejemplo del ordenador de números
| Versión | Cómo está hecho | Problema |
|---|---|---|
| v1 | La `Ventana` valida, ordena y muestra | Poco cohesiva |
| v2 | `backend.py` con funciones, la ventana **importa y llama** `procesar_input` | Cohesiva pero **acoplada** (si cambia el nombre de la función, hay que cambiar la ventana) |
| v3 | Front y back se comunican **solo con señales**, conectadas en el `main` | ✅ Cohesiva y poco acoplada |

```python
# backend.py
from PyQt6.QtCore import QObject, pyqtSignal


class Procesador(QObject):                     # hereda de QObject para tener señales
    senal_actualizar = pyqtSignal(str)          # back -> front

    def procesar_input(self, texto: str) -> None:
        texto = texto.replace(" ", "").strip(",")
        if not all(valor.isnumeric() for valor in texto.split(",")):
            self.senal_actualizar.emit("Input no válido")
            return
        numeros = sorted(int(valor) for valor in texto.split(","))
        self.senal_actualizar.emit(", ".join(str(n) for n in numeros))


# frontend.py
from PyQt6.QtCore import pyqtSignal
from PyQt6.QtWidgets import QWidget, QLineEdit, QPushButton, QLabel


class Ventana(QWidget):
    senal_procesar = pyqtSignal(str)            # front -> back

    def __init__(self) -> None:
        super().__init__()
        self.input = QLineEdit("", self)
        self.boton = QPushButton("Ordenar", self)
        self.boton.move(0, 30)
        self.resultado = QLabel("", self)
        self.resultado.setGeometry(0, 60, 300, 20)
        self.boton.clicked.connect(self.boton_clickeado)

    def boton_clickeado(self) -> None:
        self.senal_procesar.emit(self.input.text())

    def actualizar_resultado(self, texto: str) -> None:
        self.resultado.setText(texto)


# main.py: las únicas líneas que "conocen" a ambos
# ventana = Ventana(), procesador = Procesador()
# ventana.senal_procesar.connect(procesador.procesar_input)
# procesador.senal_actualizar.connect(ventana.actualizar_resultado)
# ventana.show()
```

### 14.3 Conexión entre múltiples ventanas
❌ **Error típico**: crear la otra ventana como **variable local** dentro de un método $\rightarrow$ al terminar el método, Python la descarta y la ventana **no se ve**.
```python
def abrir_otra_ventana(self):
    self.hide()
    otra_ventana = Ventana("Otra", 300, 100)    # variable local: se destruye al salir
    otra_ventana.show()
```
✅ Soluciones:
1. Guardar **siempre una referencia** a los objetos PyQt (atributo `self.otra_ventana = ...`, o en una lista que sea atributo), o crearla afuera y pasarla como argumento.
2. Mejor (ventanas independientes, bajo acoplamiento): **una señal** `senal_abrir_otra_ventana = pyqtSignal()` que la ventana emite al ocultarse, conectada en el `main` al `show` de la otra.

#### ✅ Ideas clave para recordar
- Alta cohesión + bajo acoplamiento = separar front/back y comunicarlos con señales.
- Las conexiones entre front y back se hacen en el `main`.
- Guardar referencias a todo objeto PyQt que deba seguir existiendo.

<a id="s14"></a>
# ⏱️ Semana 14 — Interfaces gráficas II (event loop, QTimer, QThread y networking)

---
# Tema 15: El *event loop* y la regla de oro


### 15.1 El *event loop*
Mientras la app está abierta, `app.exec()` repite:
1. Saca **un** evento de la cola (clic, tecla, señal emitida, orden de redibujar…).
2. Ejecuta el *slot* conectado.
3. Cuando ese método **termina**, vuelve al paso 1.

Todo esto ocurre en **un solo thread**: el principal.

### 15.2 Dos problemas
**GUI congelada**: si un slot hace `sleep(5)` (o un `recv`, o un cálculo largo), durante ese tiempo:
- la ventana no responde (no se mueve ni redimensiona),
- un `setText("trabajando...")` previo **no alcanza a verse**: `setText` solo encola un redibujado,
- los clics **no se pierden**: se procesan todos juntos al terminar.

👉 *Mientras un slot se ejecuta, el event loop está detenido*: todo slot debe terminar rápido.

**Tocar widgets desde otro thread**: puede causar `Segmentation fault`, que no se vea nada, o fallos aleatorios. Los widgets de Qt **no son thread-safe** y el event loop es código C++ que no conoce nuestros `Lock`.

### 15.3 La regla de oro
> **Solo el thread principal toca la interfaz. Los demás threads le avisan mediante señales.**

Funciona porque, si un thread secundario emite una señal conectada a un objeto que vive en el thread principal, PyQt **no** ejecuta el slot en el thread emisor: lo **encola** (*queued connection*) y lo ejecuta el event loop, como si fuera un clic.

```python
class Cuadrado(QThread):
    def __init__(self, senal_mover: pyqtSignal) -> None:
        super().__init__()
        self.senal_mover = senal_mover        # recibe la señal (no el QLabel)
        self.x, self.y = 0, 0

    def run(self) -> None:
        while True:
            sleep(0.1)
            self.x += randint(-2, 2)
            self.y += randint(-2, 2)
            self.senal_mover.emit(self.id, self.x, self.y)   # ✅ avisa, no toca el label


class MiVentana(QWidget):
    senal_mover = pyqtSignal(int, int, int)

    def __init__(self) -> None:
        super().__init__()
        self.senal_mover.connect(self.mover)

    def mover(self, id_: int, x: int, y: int) -> None:      # se ejecuta en el thread principal
        self.labels[id_].move(x, y)
```

### 15.4 ¿Necesito un thread?
| ¿Qué quiero hacer? | Ejemplo | Herramienta |
|---|---|---|
| Reaccionar rápido a una acción | Cambiar un texto al hacer clic | Un **slot normal** |
| Algo **periódico y rápido** | Mover personajes, reloj, animación | **`QTimer`** (sin threads ni concurrencia) |
| Algo que **bloquea** o tarda mucho | `recv`, cálculo de varios segundos | **Thread** (`QThread` o `threading.Thread`) que avisa con **señales** |

🚩 Señal de alerta: un `sleep` dentro de un slot congela la GUI, si es para "esperar el siguiente paso", probablemente quieres un `QTimer`.

### 15.5 ¿Por qué funciona el *hook*?
- Una excepción dentro de un slot no tiene a dónde subir (quien llamó fue el event loop en C++), PyQt la atrapa.
- Si `sys.excepthook` **sigue siendo el original** (`is sys.__excepthook__`), PyQt imprime y **aborta** (`Aborted (core dumped)`).
- Si es **otra función**, PyQt la llama y devuelve el control al loop $\rightarrow$ el programa sigue.
- PyQt revisa *si cambiaste* el hook, no qué hace: por eso `hook` puede solo llamar a `sys.__excepthook__`. Se instala siempre para que el comportamiento sea igual en todos los sistemas.
- Ojo: que siga funcionando no significa que esté bien, el slot quedó interrumpido a la mitad.

#### ✅ Ideas clave para recordar
- Slot lento = GUI congelada.
- Regla de oro: los threads emiten señales, solo el thread principal modifica widgets.
- Periódico y rápido $\rightarrow$ `QTimer`, bloqueante $\rightarrow$ thread + señales.

---
# Tema 16: `QTimer`

### 16.1 Concepto
`QTimer` (de `QtCore`) ejecuta el método conectado a `timeout` cada cierto intervalo **en el thread principal**:
- **No crea threads** $\rightarrow$ el método **sí puede modificar la interfaz** y no hay problemas de concurrencia.
- **Se repite** hasta que se detenga (a diferencia de `threading.Timer`).
- El método conectado debe ser **rápido** (si no, congela la GUI).

| Método / señal | Descripción |
|---|---|
| `QTimer(self)` | Crear (con *parent* para que no sea eliminado) |
| `timeout` | Señal emitida en cada intervalo |
| `setInterval(ms)` | Intervalo en **milisegundos** |
| `start()` / `start(ms)` | Iniciar o reiniciar |
| `stop()` | Detener |
| `isActive()` | ¿Está corriendo? |
| `setSingleShot(True)` | Ejecutar una sola vez (se puede cancelar/reiniciar) |
| `QTimer.singleShot(ms, metodo)` | Estático: una vez, sin crear objeto (no se puede cancelar) |
| `remainingTime()` | ms que faltaban (útil al pausar intervalos largos) |

```python
import datetime
from PyQt6.QtCore import QTimer
from PyQt6.QtWidgets import QWidget, QLabel


class RelojDigital(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.label = QLabel(self)
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.mostrar_hora)
        self.timer.setInterval(1000)
        self.timer.start()
        self.mostrar_hora()                      # primera vez, sin esperar 1 s

        QTimer.singleShot(3000, self.hide)       # una sola vez: esconder a los 3 s
        QTimer.singleShot(5000, self.show)       # y volver a mostrar a los 5 s

    def mostrar_hora(self) -> None:
        self.label.setText(datetime.datetime.now().strftime("%H:%M:%S"))
```
- **Varios timers**: cada objeto (p. ej. un `QObject` con su propio `QTimer`) avisa a la ventana con señales, el timer se detiene a sí mismo con `self.timer.stop()`.

### 16.2 Pausar y reanudar
Pausar = `stop()`, reanudar = `start()`. El estado (p. ej. `self.decimas`) vive en atributos y no se pierde. En un juego hay que detener **todos** los timers (conviene tenerlos en una lista). Al reanudar el intervalo parte de cero.
```python
def pausar_o_reanudar(self) -> None:
    if self.timer.isActive():
        self.timer.stop()
    else:
        self.timer.start()
```

### 16.3 *Game loop* con un solo `QTimer`
Un único timer (el *tick*, p. ej. 16 ms ≈ 60 FPS) actualiza **todos** los elementos:
- todos avanzan en el mismo instante $\rightarrow$ fácil detectar colisiones,
- pausar = detener un timer,
- todo ocurre en el thread principal $\rightarrow$ sin concurrencia.

```python
class Juego(QObject):                       # back-end
    senal_mover = pyqtSignal(int, int, int) # id, x, y
    senal_pausa = pyqtSignal(bool)

    def __init__(self) -> None:
        super().__init__()
        self.pelotas = [Pelota(i) for i in range(30)]   # Pelota = clase común (solo datos)
        self.timer = QTimer(self)
        self.timer.setInterval(16)
        self.timer.timeout.connect(self.tick)

    def tick(self) -> None:
        for pelota in self.pelotas:
            pelota.mover()
            self.senal_mover.emit(pelota.id, pelota.x, pelota.y)

# Front-end: Ventana.mover(id, x, y) mueve los labels y keyPressEvent emite
# senal_tecla_pausa si evento.key() == Qt.Key.Key_Space and not evento.isAutoRepeat()
```

#### ✅ Ideas clave para recordar
- `QTimer` corre en el thread principal: puede tocar la GUI, pero su método debe ser corto.
- Intervalos en **milisegundos**.
- Un tick para todo el juego simplifica colisiones y pausa.

---
# Tema 17: `QThread`

### 17.1 `threading.Thread` vs `QThread`
| `threading.Thread` | `QThread` |
|---|---|
| `start()` | `start()` |
| `run()` | `run()` |
| `join()` | `wait()` |
| `is_alive()` | `isRunning()` |
| `daemon = True` | **No existe**: hay que detenerlos nosotros |
| No puede definir señales (no es `QObject`) $\rightarrow$ recibe la señal por argumento | Puede definir sus **propias señales**, tiene `finished`, `started` |

> No llamar `self.thread` a un atributo: todo objeto PyQt ya tiene un método `thread()`. Usar, p. ej., `self.mi_thread`.

```python
class MiThread(QThread):
    senal_thread = pyqtSignal(str)              # señal propia

    def run(self) -> None:                      # SOLO esto corre en el thread nuevo
        for i in range(10):
            sleep(0.5)
            self.senal_thread.emit(str(i))
        self.senal_thread.emit("Status: QThread terminado")


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.mi_thread = None
        self.label = QLabel("Status: esperando", self)

    def ejecutar_thread(self) -> None:          # slot de un botón
        if self.mi_thread is None or not self.mi_thread.isRunning():
            self.mi_thread = MiThread()                         # guardar la referencia
            self.mi_thread.senal_thread.connect(self.label.setText)
            self.mi_thread.start()
```

### 17.2 Afinidad
Un objeto `QThread` **no es** el thread: **administra** uno, y como todo objeto PyQt "vive" en el thread donde fue **creado**.
1. Solo `run` se ejecuta en el thread nuevo. Cualquier otro método: si lo llamas directo, corre en quien lo llama, si está conectado a una señal, corre **donde vive el objeto** (normalmente el principal $\rightarrow$ ¡congela la GUI!).
2. El slot de una señal se ejecuta en el thread del **receptor**, si es distinto al emisor, se encola en el event loop del receptor.

**Consecuencias prácticas**
- El trabajo pesado va **dentro de `run`**. Para pedirle cosas a un thread en ejecución, usar atributos que `run` revise (bandera, `Queue`).
- Crear los objetos PyQt (`QThread`, `QTimer`, `QObject` con señales, widgets) en el **thread principal** (un `__init__` o un slot), **no dentro de `run`**: ese thread no tiene event loop, así que las señales encoladas hacia él nunca se ejecutan, y un `QTimer` creado ahí no funciona.

### 17.3 Múltiples QThreads y referencias
Guardar **siempre** una referencia (atributo o lista). Si el `QThread` queda en una variable local o se reasigna mientras corre $\rightarrow$ `QThread: Destroyed while thread '' is still running` + `Aborted`. Por eso se revisa `isRunning()` antes de crear otro.

### 17.4 Detener y cerrar limpio
Necesario al volver a un menú (si no, quedan dos juegos corriendo) y al cerrar el programa.
```python
class Contador(QThread):
    def __init__(self) -> None:
        super().__init__()
        self.activo = True                      # bandera

    def run(self) -> None:
        while self.activo:
            sleep(0.1)                          # dormir de a poco para revisar la bandera seguido
            ...

    def detener(self) -> None:
        self.activo = False


class MiVentana(QWidget):
    def closeEvent(self, evento) -> None:      # PyQt lo llama al cerrar la ventana
        self.contador.detener()
        self.contador.wait()                    # equivalente a join()
        evento.accept()
```
- Alternativa integrada: `requestInterruption()` + `isInterruptionRequested()`.
- ❌ `terminate()`: mata el thread de golpe (locks tomados, datos a medias). No usar.

### 17.5 `QMutex`
Versión PyQt del `Lock`: `mutex.lock()` / `mutex.unlock()` (en un `finally`). El `threading.Lock` también sirve, lo importante es que todos compartan **el mismo objeto**. Si se respeta la regla de oro, casi todo el estado se modifica en el thread principal y rara vez se necesita.

#### ✅ Ideas clave para recordar
- Solo `run` corre en el thread nuevo.
- Crear objetos PyQt en el thread principal, guardar referencias a los QThreads.
- Cerrar con bandera + `wait()`, nunca `terminate()`.

---
# Tema 18: Interfaces gráficas + *networking*

### 18.1 Arquitectura
```text
SERVIDOR (sin GUI)                         CLIENTE
 1 thread acepta                          ┌──────────────┐  señales  ┌───────────────┐
 1 thread por cliente  ◄── sockets ──►    │  Back-end    │ ◄───────► │  Front-end    │
                                          │  (Logica)    │           │  (Ventana)    │
                                          └──────┬───────┘           └───────────────┘
                                                 │ senal_mensaje
                                          EscucharQThread  (recv es bloqueante)
```
- El **front-end** no sabe nada de sockets: muestra y emite señales.
- El **back-end** envía (enviar mensajes pequeños es rápido $\rightarrow$ se puede hacer desde el thread principal) y procesa lo que llega.
- **Recibir** va en otro thread porque `recv` bloquea.

### 18.2 Protocolo (idéntico en ambos lados)
Es el mismo `enviar` / `recibir` / `recibir_bytes` del **Tema 10**, con la clave de que `recibir_bytes` lanza `ConnectionError` cuando `recv` retorna `b""` (la otra parte cerró) $\rightarrow$ quien llama detecta la desconexión con `try/except`.

### 18.3 Back-end del cliente
```python
class EscucharQThread(QThread):
    senal_mensaje = pyqtSignal(str)
    senal_desconectado = pyqtSignal()

    def __init__(self, socket_cliente: socket.socket) -> None:
        super().__init__()
        self.socket_cliente = socket_cliente

    def run(self) -> None:
        try:
            while True:
                mensaje = recibir(self.socket_cliente).decode("utf-8")
                self.senal_mensaje.emit(mensaje)
        except (ConnectionError, OSError):          # el servidor se cayó o cerramos el socket
            self.senal_desconectado.emit()


class Logica(QObject):
    senal_actualizar_etiqueta = pyqtSignal(str)
    senal_desconectado = pyqtSignal()

    def __init__(self, host: str, port: int) -> None:
        super().__init__()
        self.host, self.port = host, port
        self.socket_cliente = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.thread_escuchar = None

    def conectar_a_servidor(self) -> bool:
        try:
            self.socket_cliente.connect((self.host, self.port))
        except ConnectionError:
            return False
        self.thread_escuchar = EscucharQThread(self.socket_cliente)     # guardar referencia
        self.thread_escuchar.senal_mensaje.connect(self.procesar_mensaje)  # corre en el principal
        self.thread_escuchar.senal_desconectado.connect(self.senal_desconectado)
        self.thread_escuchar.start()
        return True

    def procesar_mensaje(self, mensaje: str) -> None:
        self.senal_actualizar_etiqueta.emit(mensaje)

    def mandar_comando(self, comando: str) -> None:
        try:
            enviar(self.socket_cliente, comando.encode("utf-8"))
        except (ConnectionError, OSError):
            self.senal_desconectado.emit()

    def cerrar(self) -> None:
        try:
            self.socket_cliente.shutdown(socket.SHUT_RDWR)   # despierta al recv bloqueado
        except OSError:
            pass
        self.socket_cliente.close()
        if self.thread_escuchar is not None:
            self.thread_escuchar.wait()
```

### 18.4 `main.py` y desconexiones
```python
app = QApplication([])
backend = Logica(HOST, PORT)
frontend = MainWindow()
backend.senal_actualizar_etiqueta.connect(frontend.actualizar_etiqueta)
backend.senal_desconectado.connect(frontend.mostrar_desconexion)
frontend.senal_mandar_comando.connect(backend.mandar_comando)
app.aboutToQuit.connect(backend.cerrar)       # justo antes de terminar la app
if not backend.conectar_a_servidor():
    sys.exit(1)
frontend.show()
sys.exit(app.exec())
```
- Si el servidor se cae: `recv` $\rightarrow$ `b""` $\rightarrow$ `ConnectionError` $\rightarrow$ `senal_desconectado` $\rightarrow$ la ventana avisa y deshabilita botones (no se cae ni se queda pegada).
- `OSError: Address already in use` al reiniciar el servidor $\rightarrow$ `sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)` antes del `bind` (no sirve si otro programa sigue escuchando en ese puerto).

#### ✅ Ideas clave para recordar
- Recibir en un thread, enviar desde el principal, actualizar la GUI vía señales.
- Mismo `protocolo.py` en cliente y servidor.
- `shutdown` + `close` + `wait()` para cerrar ordenado, `aboutToQuit` para engancharlo.

---
# Tema 19: Ejemplo aplicado, recetario y errores frecuentes


### 19.1 Tres back-ends para el mismo juego
| | `Thread` / `QThread` | `QTimer` |
|---|---|---|
| ¿Dónde corre `mover`? | En el thread de cada cuadrado (ciclo `while self.correr:` + `sleep`) | En el thread principal (`timeout.connect(self.mover)`) |
| ¿Cómo se detiene? | Bandera `correr = False` | `timer.stop()` |
| ¿Riesgo de concurrencia? | Sí | No |
| ¿Detener al cerrar? | Bandera + `wait()`/`join()` | `stop()` |

Mover unos píxeles es **periódico y rápido** $\rightarrow$ `QTimer` es lo más simple (y un solo timer para todo si los elementos interactúan). Se cambia de back-end modificando **un import** en `main.py`, sin tocar el front-end. `app.aboutToQuit` se conecta al `detener` del back-end.

### 19.2 Recetario
**`QMainWindow`** (ventana con esqueleto de aplicación):
```python
class MainWindow(QMainWindow):
    def __init__(self) -> None:
        super().__init__()
        salir = QAction(QIcon(None), "&Exit", self)        # QAction está en PyQt6.QtGui
        salir.setShortcut("Ctrl+Q")
        salir.setStatusTip("Salir de la aplicación")
        salir.triggered.connect(QApplication.quit)

        self.menuBar().addMenu("Archivo").addAction(salir)  # barra de menú
        self.addToolBar("Toolbar").addAction(salir)         # barra de herramientas
        self.statusBar().showMessage("Listo")               # barra de estado
        self.setCentralWidget(MiVentana())                  # cuerpo: cualquier widget
```
(En macOS la barra de menú aparece arriba en la pantalla.)

**`isAutoRepeat()`**: al mantener una tecla, `keyPressEvent` se llama muchas veces, `event.isAutoRepeat()` es `False` solo en la primera $\rightarrow$ úsalo para acciones únicas (pausar), y no lo uses para movimiento continuo.

**PyQt5 $\rightarrow$ PyQt6**: `app.exec_()` $\rightarrow$ `app.exec()` · `Qt.Key_W` $\rightarrow$ `Qt.Key.Key_W` · `Qt.LeftButton` $\rightarrow$ `Qt.MouseButton.LeftButton` · `QAction` pasó de `QtWidgets` a `QtGui`.

**Sonidos** (`PyQt6.QtMultimedia`, rutas como `QUrl.fromLocalFile(path)`):
| `QSoundEffect` | `QMediaPlayer` |
|---|---|
| Solo `.wav`, efectos cortos sin retraso | Muchos formatos (`.mp3`), música |
| `setSource(url)`, `setLoopCount(...)`, `play()` / `stop()` | Requiere `QAudioOutput` + `setAudioOutput`, `setSource(url)` (path absoluto con `os.path.abspath`) |

### 19.3 Errores frecuentes
| Síntoma | Causa probable | Solución |
|---|---|---|
| La ventana se congela | `sleep`, `recv` o ciclo largo en un slot | `QTimer` para lo periódico, thread para lo que bloquea |
| `Segmentation fault` o widgets erráticos | Un thread modifica widgets | Regla de oro: señales |
| `QThread: Destroyed while thread '' is still running` | QThread en variable local o reasignado | Guardar referencia, revisar `isRunning()` |
| Un método del QThread congela la GUI | Solo `run` corre en el thread nuevo | Trabajo pesado dentro de `run` |
| Una señal se emite pero el slot nunca corre | Receptor creado dentro de un `run` (sin event loop) | Crear objetos PyQt en el thread principal |
| Al volver al menú todo va al doble de velocidad | Threads/timers de la partida anterior siguen vivos | Detenerlos (bandera + `wait()`, o `stop()`) |
| `'QApplication' object has no attribute 'exec_'` | Código PyQt5 | `app.exec()` |
| `Aborted (core dumped)` tras una excepción | Sin hook de excepciones | Instalar `sys.excepthook` |
| No carga una imagen | Terminal en otra carpeta (ruta relativa) | `cd` a la carpeta o rutas desde `os.path.dirname(__file__)` |
| `Address already in use` | Otro servidor en el puerto / puerto recién liberado | Cerrar el otro, `SO_REUSEADDR` |
| Mensajes cortados o pegados | `recv` sin protocolo | Largo primero + chunks |

#### ✅ Ideas clave para recordar
- Periódico y rápido $\rightarrow$ `QTimer`, un solo tick si los elementos interactúan.
- `QMainWindow` = menú + toolbar + status bar + central widget.
- Usa la tabla de errores frecuentes como checklist antes de entregar.

<a id="s15"></a>
# 🧠 Semana 15 — Tópicos avanzados

---
# Tema 20: Expresiones regulares (módulo `re`)

### 20.1 Sintaxis
| Elemento | Significado | Ejemplo |
|---|---|---|
| `[abc]` | Uno de esos caracteres (dentro, los metacaracteres son literales) | `[abc$]` |
| `[a-z0-9]` | Rango | `[a-zA-Z0-9]` |
| `[^abc]` | Cualquiera **excepto** esos | |
| `.` | Cualquier carácter salvo salto de línea | `4tt.b` |
| `\d` / `\D` | Dígito / no dígito | |
| `\w` / `\W` | Alfanumérico o `_` / lo contrario | |
| `\s` / `\S` | Espacio en blanco / lo contrario | |
| `\` | Escapar un metacarácter | `\.` |
| `+` | 1 o más | `ab+c` |
| `*` | 0 o más | `ab*c` |
| `?` | 0 o 1 | `ab?c` |
| `{m}` / `{m,n}` | Exactamente m / entre m y n | `ab{3,5}c` |
| `^` / `$` | Inicio / fin del string | |
| `A\|B` | A o B | `ab+c\|de*f` |
| `( )` | Grupo de captura | `a(bc)*f` |
| `(?P<nombre> )` | Grupo con nombre | `(?P<area>\d{2})` |
| `(?: )` | Grupo **sin** captura | `(?:GGG\|GGA)` |

### 20.2 Funciones de `re`
| Función | Qué hace | Retorna |
|---|---|---|
| `re.match(p, s)` | Match **desde el inicio** (≡ `search('^...')`) | `Match` o `None` |
| `re.search(p, s)` | Primera coincidencia en **cualquier** posición | `Match` o `None` |
| `re.fullmatch(p, s)` | El string **completo** cumple | `Match` o `None` |
| `re.findall(p, s)` | Todas las coincidencias | `list` |
| `re.finditer(p, s)` | Todas, como iterador | iterador de `Match` |
| `re.sub(p, reemplazo, s)` | Reemplazar (el reemplazo puede ser una **función** que recibe el `Match`) | `str` nuevo |
| `re.split(p, s)` | Separar según el patrón | `list` |

Objeto `Match`: `span()` (inicio, fin), `group()`/`group(0)` (todo), `group(1)` (grupo posicional), `group("nombre")`, `groupdict()`. Un `Match` sirve directamente como condición (`if re.match(...)`).

> 💡 Usar *raw strings* `r"..."` para los patrones evita problemas con `\`.

In [ ]:
import re

seq = ["4tt", "4ttGb", "4ttabcabc32", "44ttkbcdag60", "3ttabc4ttyb"]
for s in seq:
    print(f"{s:13s} match={bool(re.match(r'4tt.b', s))!s:5} "
          f"search={bool(re.search(r'4tt.b', s))!s:5} fullmatch={bool(re.fullmatch(r'4tt.b', s))}")

In [ ]:
# Validación + extracción de un e-mail con grupos con nombre
PATRON_MAIL = r"(?P<username>[a-zA-Z0-9_.]+)@((?P<subdominio>seccion1|seccion2)\.)?(?P<dominio>mimail|mail)\.cl"

for correo in ["nombre.apellido@mail.cl", "nombre1010@seccion1.mimail.cl", "nombre@tumail.cl"]:
    m = re.fullmatch(PATRON_MAIL, correo)
    print(f"{correo:32s}", m.groupdict() if m else "no válido")

# RUT: versión fija y versión general (grupo repetido)
def es_rut_valido(rut: str) -> bool:
    return bool(re.fullmatch(r"\d{1,3}(\.\d{3})*-[\dkK]", rut))

for rut in ["12.224.877-2", "1.113.221-k", "344-K", "13.20.830-6", "113.427.974-12"]:
    print(f"{rut:16s}", es_rut_valido(rut))

In [ ]:
texto = "4ttabcabc32 y 44ttkbcdag60 y 3tt4ttSbc4ttyb"
print(re.findall(r"\d+", texto))                                   # todos los números
print([m.span() for m in re.finditer(r"\d+", texto)])
print(re.sub(r"\d+", "#", texto))                                  # reemplazar

complemento = {"A": "T", "T": "A", "C": "G", "G": "C"}
print(re.sub(r"[ATCG]", lambda m: complemento[m.group(0)], "ACAAGATGCC"))   # sub con función

print(re.split(r"[aeiou]", "Este es un mensaje"))
print(re.split(r"(?:GGG|GGA)", "AACGGGCAATGGATTA"))                # sin captura: no devuelve separadores

#### ✅ Ideas clave para recordar
- `match` = desde el inicio, `search` = en cualquier parte, `fullmatch` = todo el string.
- Grupos con nombre + `groupdict()` para extraer campos.
- `(?:...)` agrupa sin capturar (clave en `split`).

---
# Tema 21: Grafos y su representación

### 21.1 Definiciones
- **Grafo**: conjunto de **vértices** (nodos) y **aristas** (relaciones).
- **No dirigido**: relación simétrica (amistad en Facebook). **Dirigido**: con orientación (seguir en Instagram).
- Las **listas ligadas** (lineales) y los **árboles** (jerárquicos) son casos particulares de grafos.

### 21.2 Representaciones
| Representación | Idea | Ventaja | Desventaja |
|---|---|---|---|
| Nodos | Clase `Nodo` con lista `vecinos` | Natural, orientada a objetos | Hay que recorrer para encontrar un nodo |
| Lista de adyacencia | `dict` vértice $\rightarrow$ lista/set de vecinos | Acceso rápido a un vértice y sus vecinos, poca memoria | Saber si `x–y` existe requiere buscar en la lista |
| Matriz de adyacencia | Matriz N×N con 1/0 (simétrica si no es dirigido) | Saber si hay arista es inmediato `M[i][j]` | Ocupa N² aunque haya pocas aristas, hay que mapear índices $\leftrightarrow$ valores |

**Pesos**: se agregan guardando, p. ej., tuplas `(vecino, peso)` en la lista de adyacencia o el peso en la celda de la matriz (útil para distancias, tiempos, costos).

In [ ]:
# El mismo grafo dirigido (1->2, 2->3, 3->2, 3->4, 3->5, 4->5) en tres representaciones
class Nodo:
    def __init__(self, valor):
        self.valor = valor
        self.vecinos = []

    def agregar_vecino(self, nodo):
        self.vecinos.append(nodo)

    def __repr__(self):
        return f"[{self.valor}] -> " + ", ".join(f"[{v.valor}]" for v in self.vecinos)


nodos = {i: Nodo(i) for i in range(1, 6)}
for origen, destino in [(1, 2), (2, 3), (3, 2), (3, 4), (3, 5), (4, 5)]:
    nodos[origen].agregar_vecino(nodos[destino])
print(*nodos.values(), sep="\n")

lista_adyacencia = {1: [2], 2: [3], 3: [2, 4, 5], 4: [5], 5: []}
matriz = [[0, 1, 0, 0, 0],
          [0, 0, 1, 0, 0],
          [0, 1, 0, 1, 1],
          [0, 0, 0, 0, 1],
          [0, 0, 0, 0, 0]]
print("Vecinos de 3:", lista_adyacencia[3])
print("¿Arista 4->1?", matriz[4 - 1][1 - 1] == 1)        # índices desde 0

con_pesos = {"Santiago": [("Rancagua", 87), ("Valparaíso", 120)], "Rancagua": [("Santiago", 82)]}
print("Santiago->Rancagua:", dict(con_pesos["Santiago"])["Rancagua"], "min")

### 21.3 Operaciones básicas
`adyacentes(x, y)`, `vecinos(x)`, `agregar_vertice(x)`, `remover_vertice(x)` (¡también hay que borrarlo de las listas de los demás!), `agregar_arista(x, y)`, `remover_arista(x, y)`.

In [ ]:
class Grafo:
    def __init__(self, lista_adyacencia=None):
        self.lista_adyacencia = lista_adyacencia or dict()

    def adyacentes(self, x, y):
        return y in self.lista_adyacencia[x]

    def vecinos(self, x):
        return self.lista_adyacencia[x]

    def agregar_vertice(self, x):
        self.lista_adyacencia[x] = set()

    def remover_vertice(self, x):
        self.lista_adyacencia.pop(x, None)
        for vecinos in self.lista_adyacencia.values():
            vecinos.discard(x)

    def agregar_arista(self, x, y):
        if x in self.lista_adyacencia:
            self.lista_adyacencia[x].add(y)

    def remover_arista(self, x, y):
        self.lista_adyacencia.get(x, set()).discard(y)

    def __repr__(self):
        return "\n".join(f"{nodo}: {sorted(vecinos)}" for nodo, vecinos in self.lista_adyacencia.items())


amistades = Grafo({"Coco": {"Thor", "Luna"}, "Thor": {"Luna"}, "Luna": {"Coco"}})
amistades.agregar_vertice("Tomás")
amistades.agregar_arista("Tomás", "Luna")
amistades.remover_vertice("Thor")
print(amistades)
print("¿Coco->Luna?", amistades.adyacentes("Coco", "Luna"))

#### ✅ Ideas clave para recordar
- Lista de adyacencia (`dict`) = la representación más usada en el curso.
- Matriz: consulta inmediata, pero N² de memoria.
- Al remover un vértice hay que quitarlo también de los vecinos de los demás.

---
# Tema 22: Búsqueda en grafos: BFS y DFS


### 22.1 El problema de los ciclos
En un árbol hay **un único camino** a cada nodo, en un grafo puede haber **ciclos**, y sin registrar los visitados el recorrido se repite infinitamente. $\rightarrow$ Mantener `visitados` y **no procesar** un nodo ya visitado.

| | BFS (*Breadth-first*) | DFS (*Depth-first*) |
|---|---|---|
| Estructura | **Cola** (`deque`, `popleft`) | **Stack** (`list`, `pop`) o recursión |
| Orden | Por niveles (distancia 1, luego 2, …) | Baja por una rama hasta el fondo y retrocede |
| Útil para | Mínimo número de aristas (camino más corto **sin pesos**) | Explorar caminos, existencia de ruta, laberintos |

Ambos recorren **todo lo alcanzable** desde el inicio: si un nodo no aparece, no es alcanzable.

In [ ]:
from collections import deque

grafo = {
    "A": ["B", "C"], "B": ["D", "A"], "C": ["D"], "D": ["B"],
    "E": ["F"], "F": ["E"], "G": ["A", "F"],
}


def bfs(grafo, inicio):
    visitados = []
    cola = deque([inicio])
    while cola:
        vertice = cola.popleft()
        if vertice in visitados:          # clave: no repetir
            continue
        visitados.append(vertice)
        for vecino in grafo[vertice]:
            if vecino not in visitados:
                cola.append(vecino)
    return visitados


def dfs(grafo, inicio):
    visitados, stack = [], [inicio]
    while stack:
        vertice = stack.pop()
        if vertice in visitados:
            continue
        visitados.append(vertice)
        for vecino in grafo[vertice]:
            if vecino not in visitados:
                stack.append(vecino)
    return visitados


def dfs_recursivo(grafo, vertice, visitados=None):
    if visitados is None:                 # evitar el default mutable
        visitados = []
    visitados.append(vertice)
    for vecino in grafo[vertice]:
        if vecino not in visitados:
            dfs_recursivo(grafo, vecino, visitados)
    return visitados


for inicio in ["A", "E", "G"]:
    print(f"desde {inicio}: BFS={bfs(grafo, inicio)}  DFS={dfs(grafo, inicio)}  DFSrec={dfs_recursivo(grafo, inicio)}")

In [ ]:
# Reconstruir una ruta con BFS guardando el nodo "anterior" de cada visitado
def ruta_bfs(grafo, origen, destino):
    anterior = {origen: None}
    cola = deque([origen])
    while cola:
        actual = cola.popleft()
        if actual == destino:
            ruta = []
            while actual is not None:
                ruta.append(actual)
                actual = anterior[actual]
            return ruta[::-1]
        for vecino in grafo[actual]:
            if vecino not in anterior:
                anterior[vecino] = actual
                cola.append(vecino)
    return None


print(ruta_bfs(grafo, "G", "D"), ruta_bfs(grafo, "E", "A"))

#### ✅ Ideas clave para recordar
- BFS = cola, DFS = stack/recursión. Siempre con `visitados`.
- BFS encuentra el camino con menos aristas.
- Para la ruta, guardar el `anterior` de cada nodo y recorrer hacia atrás.

---
# Tema 23: NumPy

### 23.1 Arrays vs listas
| | Array de NumPy | Lista de Python |
|---|---|---|
| Tipos | Homogéneos (`dtype`) | Cualquiera |
| Memoria | Contigua y compacta | Referencias a objetos |
| Velocidad | Operaciones en C | Ciclos en Python |
| Operar sobre todo | **Vectorizado** (`a + b`) | Requiere un ciclo |

Convención: `import numpy as np`. NumPy infiere el `dtype` (si hay un float, todo es float) o se fija con `dtype=`.

In [ ]:
import time
import numpy as np

n = 1_000_000
lista_a, lista_b = list(range(n)), list(range(n))
inicio = time.perf_counter(), _ = [a + b for a, b in zip(lista_a, lista_b)]
t_lista = time.perf_counter() - inicio
array_a, array_b = np.arange(n), np.arange(n)
inicio = time.perf_counter(), _ = array_a + array_b
t_array = time.perf_counter() - inicio
print(f"listas: {t_lista:.4f} s | numpy: {t_array:.4f} s")

print(np.array([1, 2.5, 3]).dtype, np.array([1, 2, 3], dtype=float))
print(np.arange(0, 10, 2), np.linspace(0, 1, 5))
print(np.zeros((2, 3)), np.ones((1, 4)), np.eye(3), np.random.rand(3), sep="\n")

In [ ]:
a, b = np.array([1, 2, 3, 4]), np.array([4, 3, 2, 1])
print("a+b", a + b, "| a*b", a * b, "| a/b", a / b, "| a**2", a ** 2, "| a+10", a + 10)
print("ufuncs:", np.sqrt(a), np.exp(np.array([0, 1])))
print("comparaciones:", a > b, (a > 2) & (b < 3))          # & y | (no and / or)

m = np.array([[1, 2, 3], [4, 5, 6]])
print("m[0, 1] =", m[0, 1], "| fila 0:", m[0, :], "| columna 2:", m[:, 2])
print("suma por columnas (axis=0):", m.sum(axis=0), "| por filas (axis=1):", m.sum(axis=1))
print("filtrar con máscara:", m[m > 2])

# Broadcasting: (3,1) + (3,) -> (3,3)
print(np.array([[10], [20], [30]]) + np.array([1, 2, 3]))
print(np.concatenate((a, b)), np.split(np.arange(6), 3))

notas = np.array([5.0, 6.0, 4.0, 6.5])
print(f"suma={notas.sum()} media={notas.mean():.2f} mediana={np.median(notas)} "
      f"std={notas.std():.2f} max={notas.max()} argmax={notas.argmax()}")
idx = np.argmax(m)
fila, col = np.unravel_index(idx, m.shape)
print("posición (fila, col) del máximo de m:", (int(fila), int(col)))

#### ✅ Ideas clave para recordar
- Vectorizar en vez de iterar.
- `axis=0` opera hacia abajo (por columnas), `axis=1` hacia el lado (por filas).
- Combinar condiciones con `&` / `|` y paréntesis.

---
# Tema 24: Pandas (y un poco de Matplotlib)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Título": ["Frozen", "Toy Story", "The Lion King", "Moana", "Coco"],
    "Año": [2013, 1995, 1994, 2016, 2017],
    "Género": ["Animación"] * 5,
    "Duración": [102, 81, 88, 107, 105],
    "Calificación": [7.4, 8.3, 8.5, 7.6, 8.4],
})
df.info()
print(df.head(2))
print(df.describe())
print(df["Género"].value_counts())

### 24.1 Seleccionar y filtrar
| Quiero | Código |
|---|---|
| Una columna (`Series`) | `df["Título"]` |
| Fila por **posición** | `df.iloc[0]` · `df.iloc[:2, :4]` |
| Filas/columnas por **etiqueta** | `df.loc[:, :"Género"]` (con `loc` el final es **inclusivo**) |
| Filtrar | `df[df["Calificación"] > 8]` (combinar con `&` / `\|`) |
| Índice del máximo | `df.loc[df["Calificación"].idxmax()]` |

In [ ]:
print(df.iloc[0]["Título"], "|", df.iloc[:2, :2].values.tolist())
print(df.loc[:, :"Año"].head(2))
top = df[(df["Calificación"] > 8) & (df["Año"] > 2000)]
print(top)
print("Mejor:", df.loc[df["Calificación"].idxmax(), "Título"])

# Modificar
df = pd.concat([df, pd.DataFrame([{"Título": "Encanto", "Año": 2021, "Género": "Animación",
                                   "Duración": 102, "Calificación": 7.2}])], ignore_index=True)
df["Calificación Ajustada"] = df["Calificación"] * 10                 # columna calculada
df["Categoría"] = "Regular"
df.loc[df["Calificación"] >= 8.0, "Categoría"] = "Excelente"          # asignar con loc
print(df.groupby("Categoría")["Calificación"].mean())

# Combinar con join (mismo índice)
finanzas = pd.DataFrame({
    "Título": ["Frozen", "Toy Story", "The Lion King", "Moana", "Coco", "Encanto"],
    "Presupuesto": [150, 30, 45, 150, 175, 120],
    "Recaudación": [1276.5, 373.6, 968.5, 643.3, 807.1, 256.8],
})
completo = df.set_index("Título").join(finanzas.set_index("Título"))
print(completo[["Calificación", "Presupuesto", "Recaudación"]])

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(completo.index, completo["Calificación"], color="skyblue")
ax1.set_title("Calificación IMDb"), ax1.set_xlabel("Película"), ax1.set_ylabel("Calificación")
ax1.tick_params(axis="x", rotation=45)
ax2.scatter(completo["Presupuesto"], completo["Recaudación"], color="purple")
for titulo, fila in completo.iterrows():
    ax2.annotate(titulo, (fila["Presupuesto"], fila["Recaudación"]),
                 textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
ax2.set_title("Presupuesto vs recaudación (MM USD)"), ax2.set_xlabel("Presupuesto"), ax2.set_ylabel("Recaudación")
plt.tight_layout(), plt.show()

# CSV: guardar y volver a cargar
ruta = Path("resumen_tmp_peliculas.csv")
completo.to_csv(ruta)                 # index=False si no se quiere guardar el índice
print(pd.read_csv(ruta).head(3))
ruta.unlink()

#### ✅ Ideas clave para recordar
- `iloc` = posiciones (fin exclusivo), `loc` = etiquetas (fin inclusivo).
- Filtrar con máscaras booleanas, asignar con `df.loc[mascara, col] = valor`.
- `groupby(col)[otra].mean()` para agregar, `set_index` + `join` para combinar.

---
# 🏁 Fin del resumen
**Cómo se conectan los temas:**
- Serialización (JSON/pickle) + bytes $\rightarrow$ lo que viaja por los **sockets** (con un **protocolo** de largo + chunks).
- Sockets bloquean $\rightarrow$ **threads** en el servidor (uno por cliente) y en el cliente (uno que escucha).
- En una GUI, ese thread que escucha **no toca widgets**: emite **señales** (regla de oro), lo periódico va con **`QTimer`**.
- **Front-end / back-end** + señales = alta cohesión y bajo acoplamiento.
- Semana 15: herramientas para procesar datos (regex, grafos con BFS/DFS, NumPy y Pandas).